In [330]:
import pandas as pd
import numpy as np
from math import acos, degrees
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [334]:
FILENAME = 'DJI_0056.csv'
FILE_PATH = f'csv_files/{FILENAME}'
MAP_KEYPOINTS = {0:'nose', 1:'leye', 2:'reye', 3:'lear', 4:'rear', 5:'lshoulder', 6:'rshoulder', 7:'lelbow', 8:'relbow',
                 9:'lwrist', 10:'rwrist', 11:'lhip', 12:'rhip', 13:'lknee', 14:'rknee', 15:'lankle', 16:'rankle'}
SPEED_CALC_INTERVAL = 29
MAP_DIRECTION = {-1: 'UNKNOWN', 0: 'LEFT', 1: 'RIGHT', 2: 'UP', 3: 'DOWN'}


In [335]:
df = pd.read_csv(FILE_PATH, header=None)
print(df.shape)
for i in range(3,df.shape[1]):
    df[i] =  df[i].apply(lambda x: x.replace('[','').replace(']','')) 

(3427, 37)


In [336]:
# only applicable when keypoint 0 is column '3'
# 0->3
# 1->5
# 2->7

In [337]:
def distance(ax,ay,bx,by):
    return np.sqrt((ax-bx)**2+(ay-by)**2)

def angle(ax,ay,bx,by,cx,cy):
    import math
    ang = degrees(math.atan2(cy-by, cx-bx) - math.atan2(ay-by, ax-bx))
    return ang + 360 if ang < 0 else ang


In [338]:
distance_features = []
for i in range(17):
    for j in range(i+1,17):
        # x, y
        ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
        bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
        colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j]
        df[colname] = distance(ax,ay,bx,by)
        distance_features.append(colname)

In [386]:
df.head()

,red_marker,direction,speed,nose_leye,nose_reye,nose_lear,nose_rear,nose_lshoulder,nose_rshoulder,nose_lelbow,...,rhip_lknee_rknee,rhip_lknee_lankle,rhip_lknee_rankle,rhip_rknee_lankle,rhip_rknee_rankle,rhip_lankle_rankle,lknee_rknee_lankle,lknee_rknee_rankle,lknee_lankle_rankle,rknee_lankle_rankle
0,False,1,-1.0,5.511833,5.891233,8.011920,8.019399,38.319322,37.362858,45.583476,...,322.218589,248.055058,315.190208,81.769323,154.854973,316.144842,7.291730,80.377380,254.328016,352.872755
1,False,1,-1.0,5.550607,5.944650,8.135032,8.110572,38.326463,37.268221,45.447915,...,321.553718,244.870419,313.960051,83.126680,154.884013,315.359779,7.617556,79.374890,256.657436,352.356580
2,False,1,-1.0,5.343683,5.746487,7.527912,7.518215,38.130795,37.382750,45.691004,...,321.995848,242.641745,314.336143,83.851498,153.817096,316.036566,7.746401,77.711999,259.473755,352.373251
3,False,1,-1.0,5.539007,5.913945,8.099385,8.072868,38.221098,37.362269,45.806139,...,322.187813,243.425519,314.814576,83.286693,153.054190,316.342642,7.405320,77.172816,258.822912,352.655299
4,False,1,-1.0,5.360990,5.758592,7.563576,7.593624,38.039641,37.357407,45.736504,...,322.798965,244.866940,315.841919,82.731210,152.873816,317.167607,7.230910,77.373517,258.219404,353.056468


In [340]:
angle_features = []
for i in range(17):
    for j in range(i+1,17):
        for k in range(j+1, 17):
            # x, y
            ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
            bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
            cx, cy = df[k*2+3].astype(float),df[k*2+4].astype(float)
            colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j] + '_' + MAP_KEYPOINTS[k]
            angle_array = []
            for idx in range(len(ax)):
                angle_array.append(angle(ax[idx],ay[idx],bx[idx],by[idx],cx[idx],cy[idx]))
            df[colname] = angle_array
            angle_features.append(colname)

In [341]:
df.rename(columns={0:'red_marker', 1:'direction', 2:'speed'}, inplace=True)
df = df[['red_marker','direction','speed']+distance_features+angle_features]


In [342]:
groups = df.groupby(np.arange(len(df.index))//SPEED_CALC_INTERVAL)
rows = []
for (frameno, frame) in groups:
    red_marker_ = np.any(frame['red_marker'])
    direction_ = max(frame['direction'])
    speed_ = max(frame['speed'])
    mean_ = list(frame.iloc[:,3:].mean())
    rows.append([red_marker_, direction_, speed_] + mean_)
    # print(mean_)
    # break
    # print(red_marker_)

In [ ]:
final_df = pd.DataFrame(rows, columns=['red_marker','direction','speed']+distance_features+angle_features)
final_df['direction'] = final_df['direction'].map(lambda x: MAP_DIRECTION[x])

final_df['time'] = np.arange(0,final_df.shape[0]*0.5,0.5)
final_df['time'] = final_df['time'].map(lambda x: f'{x//60}:{x%60}')
final_df = final_df[['time','red_marker','direction','speed']+distance_features+angle_features]
final_df['speed_pct_change'] = final_df['speed'].pct_change()

In [388]:
for i in angle_features:
    final_df[i] = np.unwrap(final_df[i], period=360)


In [389]:
distance_features_pct = [i + '_pct' for i in distance_features]
for i,j in zip(distance_features_pct,distance_features):
    final_df[i] = final_df[j].pct_change()


angle_features_pct = [i + '_pct' for i in angle_features]
for i,j in zip(angle_features_pct,angle_features):
    final_df[i] = final_df[j].pct_change()

In [390]:
def find_top_correlation(row):
    idx_max = row.argmax()
    value_max = row[idx_max]
    colname_max = row.index[idx_max]

    idx_min = row.argmin()
    value_min = row[idx_min]
    colname_min = row.index[idx_min]
    return f'{colname_max}:{value_max}, {colname_min}:{value_min}'

In [391]:
# final_df['correlation'] = final_df[distance_features_pct+angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['distance_correlation'] = final_df[distance_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['angle_correlation'] = final_df[angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)

C:\Users\Admin\AppData\Local\Temp\ipykernel_20460\202334822.py:2: FutureWarning: The behavior of Series.argmax/argmin with skipna=False and NAs, or with all-NAs is deprecated. In a future version this will raise ValueError.
  idx_max = row.argmax()
C:\Users\Admin\AppData\Local\Temp\ipykernel_20460\202334822.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  value_max = row[idx_max]
C:\Users\Admin\AppData\Local\Temp\ipykernel_20460\202334822.py:6: FutureWarning: The behavior of Series.argmax/argmin with skipna=False and NAs, or with all-NAs is deprecated. In a future version this will raise ValueError.
  idx_min = row.argmin()
C:\Users\Admin\AppData\Local\Temp\ipykernel_20460\202334822.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys w

In [393]:
final_df['angle_correlation']

0      rknee_lankle_rankle_pct:nan, rknee_lankle_rank...
1      reye_rknee_rankle_pct:14.008953587788891, nose...
2      leye_lknee_lankle_pct:11.643255895646478, lelb...
3      lshoulder_lhip_lankle_pct:10.330697381223416, ...
4      lelbow_lhip_lknee_pct:37.443849933785096, lear...
                             ...                        
114    lear_lelbow_lankle_pct:1.687150413264992, lear...
115    lshoulder_relbow_lknee_pct:0.917508794810705, ...
116    nose_rwrist_lknee_pct:14.554530469787869, lelb...
117    relbow_lknee_rankle_pct:6.075079282707874, rea...
118    rwrist_rhip_rankle_pct:11.32772234434248, nose...
Name: angle_correlation, Length: 119, dtype: object

In [394]:
final_df[['time','red_marker','direction','speed','speed_pct_change','distance_correlation','angle_correlation']].to_csv('test.csv')

In [ ]:
# print(final_df.shape)
# final_df.to_csv(f'cleaned_csv_files/{FILENAME}')


(30, 820)
